# Session 2: Working with MongoDB Databases (CRUD)

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-02_create_read.py`

In [2]:
# Session 2 · Create and Read
from store_db import get_db, show

db = get_db()
db.drop_collection("staff")                        # start clean

# CREATE ------------------------------------------------------------
r1 = db.staff.insert_one({"_id": "E01", "name": "Zara", "role": "Cashier", "salary": 60000})
print("insert_one ->", r1.inserted_id)
r2 = db.staff.insert_many([
    {"_id": "E02", "name": "Kamran", "role": "Technician", "salary": 85000, "skills": ["laptops", "phones"]},
    {"_id": "E03", "name": "Mehwish", "role": "Manager", "salary": 150000},
    {"_id": "E04", "name": "Danish", "role": "Technician", "salary": 78000, "skills": ["phones"]},
])
print("insert_many ->", r2.inserted_ids)

# READ --------------------------------------------------------------
print("\n1) find_one by _id")
show(db.staff.find_one({"_id": "E03"}))

print("\n2) equality filter + projection (only name and salary)")
show(db.staff.find({"role": "Technician"}, {"_id": 0, "name": 1, "salary": 1}))

print("\n3) products: Accessories, sort by price descending, top 3")
show(db.products.find({"category": "Accessories"}, {"name": 1, "price": 1})
                .sort("price", -1).limit(3))

print("\n4) paging: skip 3, limit 3 (page 2 of products ordered by _id)")
show(db.products.find({}, {"name": 1}).sort("_id", 1).skip(3).limit(3))

print("\n5) query inside an embedded document and an array")
show(db.products.find({"specs.brand": "Logi", "tags": "office"}, {"name": 1, "specs.brand": 1}))

print("\n6) counting")
print("all customers:", db.customers.count_documents({}),
      "| Karachi customers:", db.customers.count_documents({"city": "Karachi"}))
print("distinct cities:", sorted(db.customers.distinct("city")))

insert_one -> E01
insert_many -> ['E02', 'E03', 'E04']

1) find_one by _id
{"_id": "E03", "name": "Mehwish", "role": "Manager", "salary": 150000}
(1 document)

2) equality filter + projection (only name and salary)
{"name": "Kamran", "salary": 85000}
{"name": "Danish", "salary": 78000}
(2 documents)

3) products: Accessories, sort by price descending, top 3
{"name": "Mechanical Keyboard", "price": 7500, "_id": 102}
{"name": "Webcam HD", "price": 4500, "_id": 107}
{"name": "USB-C Hub", "price": 4200, "_id": 104}
(3 documents)

4) paging: skip 3, limit 3 (page 2 of products ordered by _id)
{"name": "USB-C Hub", "_id": 104}
{"name": "Noise-Cancelling Headphones", "_id": 105}
{"name": "Bluetooth Speaker", "_id": 106}
(3 documents)

5) query inside an embedded document and an array
{"name": "Wireless Mouse", "specs": {"brand": "Logi"}, "_id": 101}
{"name": "Webcam HD", "specs": {"brand": "Logi"}, "_id": 107}
(2 documents)

6) counting
all customers: 6 | Karachi customers: 3
distinct cities:

## Example: `session-02_update_delete.py`

In [3]:
# Session 2 · Update and Delete
from pymongo import ReturnDocument
from store_db import get_db, show

db = get_db()

print("1) update_one with $set and $inc")
r = db.products.update_one({"_id": 104}, {"$set": {"stock": 25}, "$inc": {"price": 300}})
print("matched", r.matched_count, "modified", r.modified_count)
show(db.products.find({"_id": 104}, {"name": 1, "price": 1, "stock": 1}))

print("\n2) update_many: Rs 500 off every Audio product")
r = db.products.update_many({"category": "Audio"}, {"$inc": {"price": -500}})
print("modified", r.modified_count)
show(db.products.find({"category": "Audio"}, {"name": 1, "price": 1}))

print("\n3) upsert: update if exists, otherwise insert")
r = db.customers.update_one({"email": "noor@example.pk"},
                            {"$set": {"name": "Noor Fatima", "city": "Hyderabad"}}, upsert=True)
print("matched", r.matched_count, "upserted_id", r.upserted_id is not None)

print("\n4) replace_one swaps the whole document (except _id)")
db.customers.replace_one({"_id": 6}, {"name": "Fahad Raza", "city": "Multan"})
show(db.customers.find_one({"_id": 6}))

print("\n5) find_one_and_update returns the document in one atomic step")
doc = db.products.find_one_and_update({"_id": 101}, {"$inc": {"stock": -1}},
                                      projection={"name": 1, "stock": 1},
                                      return_document=ReturnDocument.AFTER)
show(doc)

print("\n6) delete_one and delete_many")
print("cancelled orders deleted:", db.orders.delete_many({"status": "Cancelled"}).deleted_count)
print("one pending order deleted:", db.orders.delete_one({"status": "Pending"}).deleted_count)
print("orders left:", db.orders.count_documents({}))

1) update_one with $set and $inc
matched 1 modified 1
{"name": "USB-C Hub", "price": 4500, "stock": 25, "_id": 104}
(1 document)

2) update_many: Rs 500 off every Audio product
modified 2
{"name": "Noise-Cancelling Headphones", "price": 18000, "_id": 105}
{"name": "Bluetooth Speaker", "price": 6000, "_id": 106}
(2 documents)

3) upsert: update if exists, otherwise insert
matched 0 upserted_id True

4) replace_one swaps the whole document (except _id)
{"_id": 6, "name": "Fahad Raza", "city": "Multan"}
(1 document)

5) find_one_and_update returns the document in one atomic step
{"name": "Wireless Mouse", "stock": 39, "_id": 101}
(1 document)

6) delete_one and delete_many
cancelled orders deleted: 1
one pending order deleted: 1
orders left: 8


## Example: `session-02_bulk_write.py`

In [4]:
# Session 2 · Bulk writes and handling duplicate keys
from pymongo import InsertOne, UpdateOne, DeleteOne
from pymongo.errors import BulkWriteError, DuplicateKeyError
from store_db import get_db

db = get_db()

ops = [
    InsertOne({"_id": 111, "name": "Graphics Tablet", "category": "Accessories", "price": 12000, "stock": 4}),
    UpdateOne({"_id": 102}, {"$set": {"stock": 30}}),
    DeleteOne({"_id": 104}),
]
res = db.products.bulk_write(ops)
print("bulk result -> inserted:", res.inserted_count, "modified:", res.modified_count, "deleted:", res.deleted_count)

# _id must be unique
try:
    db.products.insert_one({"_id": 101, "name": "Duplicate mouse"})
except DuplicateKeyError:
    print("DuplicateKeyError: _id 101 already exists")

# ordered=True (default) stops at the first error; ordered=False carries on
docs = [{"_id": 201}, {"_id": 101}, {"_id": 202}]
for ordered in (True, False):
    db.products.delete_many({"_id": {"$in": [201, 202]}})
    try:
        db.products.insert_many(docs, ordered=ordered)
    except BulkWriteError as e:
        print(f"ordered={ordered}: inserted {e.details['nInserted']} of 3, errors: {len(e.details['writeErrors'])}")

bulk result -> inserted: 1 modified: 1 deleted: 1
DuplicateKeyError: _id 101 already exists
ordered=True: inserted 1 of 3, errors: 1
ordered=False: inserted 2 of 3, errors: 1
